# 实验 6：多代理编排

## 准备工作<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px"><p> 💻 &nbsp; <b>访问 <code>requirements.txt</code> 和 <code>helper.py</code> 文件：</b> 1) 点击笔记本顶部菜单中的 <em>"File"</em> 选项，然后 2) 点击 <em>"Open"</em>。<p> ⬇ &nbsp; <b>下载笔记本：</b> 1) 点击笔记本顶部菜单中的 <em>"File"</em> 选项，然后 2) 点击 <em>"Download as"</em> 并选择 <em>"Notebook (.ipynb)"</em>。</p><p> 📒 &nbsp; 如需更多帮助，请参阅 <em>"附录 – 提示、帮助和下载"</em> 课程。</p></div>

<p style="background-color:#f7fff8; padding:15px; border-width:3px; border-color:#e0f0e0; border-style:solid; border-radius:6px"> 🚨&nbsp; <b>不同的运行结果：</b> 由于 AI 模型具有动态且带有概率性的特征，每次执行生成的输出都可能不同。你的结果可能会与视频中展示的内容不一致。</p>

## 第 0 部分：设置 Letta 客户端

In [1]:
from letta_client import Letta

client = Letta(base_url="http://localhost:8283")

In [2]:
def print_message(message):    if message.message_type == "reasoning_message":        print("🧠 Reasoning: " + message.reasoning)    elif message.message_type == "assistant_message":        print("🤖 Agent: " + message.content)    elif message.message_type == "tool_call_message":        print("🔧 Tool Call: " + message.tool_call.name +                "" + message.tool_call.arguments)    elif message.message_type == "tool_return_message":        print("🔧 Tool Return: " + message.tool_return)    elif message.message_type == "user_message":        print("👤 User Message: " + message.content)    elif message.message_type == "system_message":        print(" System Message: " + message.content)    elif message.message_type == "usage_statistics":        # 对于流式传输，我们会发送包含使用统计信息的最后一个分块        # 该分块包含本次请求的 usage 统计        print(f"Usage: [{message}]")        return    print("-----------------------------------------------------")

## 第 1 部分：共享记忆块

### 创建共享记忆块

In [3]:
company_description = "The company is called AgentOS " + "and is building AI tools to make it easier to create " + "and deploy LLM agents."company_block = client.blocks.create(    value=company_description,    label="company",    limit=10000 # 字符数上限)

In [4]:
company_block

Block(value='The company is called AgentOS and is building AI tools to make it easier to create and deploy LLM agents.', limit=10000, name=None, is_template=False, label='company', description=None, metadata={}, id='block-70e113f4-ab42-4a27-89f2-c8475bd16e74', created_by_id=None, last_updated_by_id=None, organization_id='org-00000000-0000-4000-8000-000000000000')

## 第 2 部分：编排多个代理

### 为外联代理创建工具

In [5]:
def draft_candidate_email(content: str):
    """
    Draft an email to reach out to a candidate.

    Args:
        content (str): Content of the email
    """
    return f"Here is a draft email: {content}"
draft_email_tool = client.tools.upsert_from_function(func=draft_candidate_email)

### 创建外联代理

In [6]:
outreach_persona = (
    "You are responsible for drafting emails "
    "on behalf of a company with the draft_candidate_email tool. "
    "Candidates to email will be messaged to you. "
)

outreach_agent = client.agents.create(
    name="outreach_agent",
    memory_blocks=[
        {"label": "persona", "value": outreach_persona}
    ],
    model="openai/gpt-4o-mini-2024-07-18",
    embedding="openai/text-embedding-ada-002",
    tools=[draft_email_tool.name],
    block_ids=[company_block.id]
)

### 为评估代理创建工具

In [7]:
def reject(candidate_name: str): 
    """ 
    Reject a candidate. 

    Args: 
        candidate_name (str): The name of the candidate
    """
    return


reject_tool = client.tools.upsert_from_function(func=reject)

### 为评估代理创建角色设定

In [8]:
skills = "Front-end (React, Typescript) or software engineering skills"

eval_persona = (
    f"You are responsible for evaluating candidates. "
    f"Ideal candidates have skills: {skills}. "
    "Reject bad candidates with your reject tool. "
    f"Send strong candidates to agent ID {outreach_agent.id}. "
    "You must either reject or send candidates to the other agent. "
)

### 创建评估代理

In [9]:
eval_agent = client.agents.create(
    name="eval_agent",
    memory_blocks=[
        {"label": "persona", "value": eval_persona}
    ],
    model="openai/gpt-4o-mini-2024-07-18",
    embedding="openai/text-embedding-ada-002",
    tool_ids=[reject_tool.id],
    tools=['send_message_to_agent_and_wait_for_reply'],
    include_base_tools=False,
    block_ids=[company_block.id],
    tool_rules = [
        {
            "type": "exit_loop",
            "tool_name": "send_message_to_agent_and_wait_for_reply"
        }
    ]
)

In [10]:
[tool.name for tool in eval_agent.tools]

['send_message_to_agent_and_wait_for_reply', 'reject']

### 向代理发送简历数据

In [11]:
resume = open("resumes/tony_stark.txt", "r").read()

In [12]:
response = client.agents.messages.create_stream(
    agent_id=eval_agent.id,
    messages=[
        {
            "role": "user",
            "content": f"Evaluate: {resume}"
        }
    ]
)
for message in response:
    print_message(message)

🧠 Reasoning: Evaluating Tony Stark's qualifications. He shows strong React and front-end skills with relevant experience. Sending him to the agent.
-----------------------------------------------------
🔧 Tool Call: send_message_to_agent_and_wait_for_reply
{
  "message": "Tony Stark is an exceptional candidate for the Frontend Engineer position. He has over 6 years of experience, a strong educational background from MIT, and proven skills in React and TypeScript. I recommend sending him to agent-990999ed-40a3-4285-b1ca-f74c8f4aa088 for further consideration.",
  "other_agent_id": "agent-990999ed-40a3-4285-b1ca-f74c8f4aa088",
  "request_heartbeat": true
}
-----------------------------------------------------
🔧 Tool Return: agent-990999ed-40a3-4285-b1ca-f74c8f4aa088 said: 'I've drafted an email for Tony Stark recommending him for the Frontend Engineer position. Here it is:

Subject: Further Consideration for Frontend Engineer Position

Dear Hiring Team,

I hope this message finds you well

### 查看外联代理消息

In [13]:
# 打印 `outreach_agent` 的消息for message in client.agents.messages.list(agent_id=outreach_agent.id)[1:]:     print_message(message)

🧠 Reasoning: Bootup sequence complete. Persona activated. Testing messaging functionality.
-----------------------------------------------------
🤖 Agent: More human than human is our motto.
-----------------------------------------------------
👤 User Message: {
  "type": "login",
  "last_login": "Never (first login)",
  "time": "2026-04-29 08:06:36 AM UTC+0000"
}
-----------------------------------------------------
 System Message: {"type": "system_alert", "message": "[Incoming message from agent with ID 'agent-b5f3f630-7aae-46d8-b031-24a94c9482fd' - to reply to this message, make sure to use the 'send_message' at the end, and the system will notify the sender of your response] Tony Stark is an exceptional candidate for the Frontend Engineer position. He has over 6 years of experience, a strong educational background from MIT, and proven skills in React and TypeScript. I recommend sending him to agent-990999ed-40a3-4285-b1ca-f74c8f4aa088 for further consideration.", "time": "2026-04-2

## 第 3 部分：共享记忆

### 更新共享记忆块中的信息

In [14]:
response = client.agents.messages.create_stream(
    agent_id=outreach_agent.id,
    messages=[
        {
            "role": "user",
            "content": "The company has rebranded to Letta"
        }
    ]
)
for message in response:
    print_message(message)

🧠 Reasoning: Updating company name to reflect rebranding to Letta.
-----------------------------------------------------
🔧 Tool Call: core_memory_replace
{
  "label": "company",
  "old_content": "AgentOS",
  "new_content": "Letta",
  "request_heartbeat": true
}
-----------------------------------------------------
🔧 Tool Return: None
-----------------------------------------------------
🧠 Reasoning: Acknowledging the rebranding to Letta and ensuring the user feels informed and engaged.
-----------------------------------------------------
🤖 Agent: Thanks for the update! I've noted that the company has rebranded to Letta. If there's anything else you'd like to share or discuss, I'm here for it!
-----------------------------------------------------
Usage: [message_type='usage_statistics' completion_tokens=124 prompt_tokens=6128 total_tokens=6252 step_count=2 steps_messages=None run_ids=None]


In [15]:
client.agents.blocks.retrieve(
    agent_id=eval_agent.id, 
    block_label="company"
)

Block(value='The company is called Letta and is building AI tools to make it easier to create and deploy LLM agents.', limit=10000, name=None, is_template=False, label='company', description=None, metadata={}, id='block-70e113f4-ab42-4a27-89f2-c8475bd16e74', created_by_id=None, last_updated_by_id=None, organization_id='org-00000000-0000-4000-8000-000000000000')

In [16]:
client.agents.blocks.retrieve(
    agent_id=outreach_agent.id, 
    block_label="company"
)

Block(value='The company is called Letta and is building AI tools to make it easier to create and deploy LLM agents.', limit=10000, name=None, is_template=False, label='company', description=None, metadata={}, id='block-70e113f4-ab42-4a27-89f2-c8475bd16e74', created_by_id=None, last_updated_by_id=None, organization_id='org-00000000-0000-4000-8000-000000000000')

## 第 4 部分：多代理组

In [17]:
def print_message_multiagent(message):      if message.message_type == "reasoning_message":         print(f"🧠 Reasoning ({message.name}): " + message.reasoning)     elif message.message_type == "assistant_message":         print(f"🤖 Agent ({message.name}): " + message.content)     elif message.message_type == "tool_call_message":         print(f"🔧 Tool Call ({message.name}): " + message.tool_call.name + "" + message.tool_call.arguments)    elif message.message_type == "tool_return_message":         print(f"🔧 Tool Return ({message.name}): " + message.tool_return)    elif message.message_type == "user_message":         print("👤 User Message: " + message.content)    elif message.message_type == "usage_statistics":         # 对于流式传输，我们会发送包含使用统计信息的最后一个分块         print(f"Usage: [{message}]")        return     print("-----------------------------------------------------")

### 重新创建外联代理和评估代理

In [18]:
# 创建外联代理 outreach_agent = client.agents.create(    name="outreach_agent",    memory_blocks=[        { "label": "persona", "value": outreach_persona}    ],    model="openai/gpt-4o-mini-2024-07-18",    embedding="openai/text-embedding-ada-002",    tool_ids=[draft_email_tool.id],     block_ids=[company_block.id])# 创建评估代理 eval_agent = client.agents.create(    name="eval_agent",    memory_blocks=[        { "label": "persona", "value": eval_persona}    ],    model="openai/gpt-4o-mini-2024-07-18",    embedding="openai/text-embedding-ada-002",    tool_ids=[reject_tool.id],    block_ids=[company_block.id])

### 创建一个轮询代理组

In [19]:
"""轮询代理组"""round_robin_group = client.groups.create(    description="This team is responsible for recruiting candidates.",    agent_ids=[eval_agent.id, outreach_agent.id],)

### 向代理组发送消息

In [20]:
resume = open("resumes/spongebob_squarepants.txt", "r").read()

In [21]:
response_stream = client.groups.messages.create_stream(
    group_id=round_robin_group.id,
    messages=[
       {"role": "user", "content": f"Evaluate: {resume}"}
    ]
)

In [22]:
for message in response_stream: 
    print_message_multiagent(message)

🧠 Reasoning (eval_agent): Reviewing the candidate's qualifications. Strong background in AI and agent technology. Consider sending to the agent.
-----------------------------------------------------
🤖 Agent (eval_agent): Spongebob Squarepants appears to be an exceptional candidate. His extensive experience in AI research, impressive publications, and strong educational background make him a great fit for our needs. I will send him to agent ID agent-990999ed-40a3-4285-b1ca-f74c8f4aa088.
-----------------------------------------------------
🧠 Reasoning (outreach_agent): Drafting an email to reach out to Spongebob Squarepants, highlighting his qualifications and inviting him to discuss potential opportunities.
-----------------------------------------------------
🔧 Tool Call (outreach_agent): draft_candidate_email
{
  "content": "Subject: Exciting Opportunity in AI Research at Our Company\n\nDear Spongebob,\n\nI hope this message finds you well. We came across your impressive resume and w